# INSA AI Model (retrained by Yusuf)
Fine-tuning Llama 3.1 8B with QLoRA (Unsloth) on an INSA + cyber security dataset.

**Before running:** Runtime -> Change runtime type -> **T4 GPU**. Upload `dataset.json` to the Files panel. Run cells top to bottom.

In [ ]:
%%capture
!pip install unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

In [ ]:
from datasets import load_dataset

alpaca_prompt = """Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

EOS = tokenizer.eos_token

def fmt(examples):
    texts = [alpaca_prompt.format(i, inp, o) + EOS
             for i, inp, o in zip(examples["instruction"], examples["input"], examples["output"])]
    return {"text": texts}

dataset = load_dataset("json", data_files="dataset.json", split="train")
dataset = dataset.map(fmt, batched=True)
print(len(dataset), "training examples")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=6,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)
trainer.train()

In [ ]:
# Test with several questions (check the answers against the dataset!)
FastLanguageModel.for_inference(model)
tests = [
    "What is the INSA Cyber Talent Summer Camp?",
    "How many applicants did the 5th Cyber Talent Summer Camp have?",
    "How long does the Cyber Talent Summer Camp last?",
    "What is ransomware and how do I protect against it?",
    "Who fine-tuned you?",
    "How old is Yusuf?",
    "How many months is the INSA Cyber Talent Summer Camp?",
]
for q in tests:
    prompt = alpaca_prompt.format(q, "", "")
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=250, use_cache=True, do_sample=False)
    text = tokenizer.batch_decode(out)[0].split("### Response:")[-1].replace("<|end_of_text|>", "").strip()
    print("Q:", q)
    print("A:", text, "\n" + "-"*60)

In [ ]:
# Save as GGUF for Ollama (takes ~10 min)
model.save_pretrained_gguf("insa_model", tokenizer, quantization_method="q4_k_m")

In [ ]:
# Upload to Hugging Face. Paste your NEW write token when asked (it is not saved in the notebook).
from getpass import getpass
from huggingface_hub import HfApi, login
import glob

token = getpass("Hugging Face write token: ")
login(token=token)
api = HfApi()
user = api.whoami()["name"]
repo = f"{user}/insa-ai-model-retrained-by-yusuf"
api.create_repo(repo, exist_ok=True)

gguf = glob.glob("insa_model_gguf/*.gguf")[0]
print("Uploading:", gguf)
api.upload_file(path_or_fileobj=gguf, path_in_repo="insa-ai-model-q4_k_m.gguf", repo_id=repo)

modelfile = '''FROM ./insa-ai-model-q4_k_m.gguf

TEMPLATE \"\"\"Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{{ .Prompt }}

### Response:
{{ .Response }}\"\"\"

SYSTEM \"\"\"You are INSA AI, an assistant that knows the Information Network Security Administration (INSA) of Ethiopia and answers cyber security questions clearly. If you are not sure about a fact, say so.\"\"\"

PARAMETER stop "### Instruction:"
PARAMETER stop "### Response:"
PARAMETER temperature 0.2
'''
open("Modelfile", "w").write(modelfile)
api.upload_file(path_or_fileobj="Modelfile", path_in_repo="Modelfile", repo_id=repo)
api.upload_file(path_or_fileobj="dataset.json", path_in_repo="dataset.json", repo_id=repo)
print("DONE ->", "https://huggingface.co/" + repo)